In [116]:
import torch
import torch.nn as nn
import torchvision.models
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision import models

from tqdm import tqdm
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import numpy as np

import os
from time import time

### Get the data

In [3]:
import gdown
url = 'https://drive.google.com/uc?id=10f1H2T-5W-BiqabHHtlZ4ASs19TZmg8R'
output = 'data.zip'
gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=10f1H2T-5W-BiqabHHtlZ4ASs19TZmg8R
From (redirected): https://drive.google.com/uc?id=10f1H2T-5W-BiqabHHtlZ4ASs19TZmg8R&confirm=t&uuid=3fb2a1ba-1674-4785-b581-89674a7fdb51
To: /kaggle/working/data.zip
100%|██████████| 979M/979M [00:05<00:00, 176MB/s] 


'data.zip'

In [ ]:
!unzip data.zip

### Utilities (0.5 point)

Complete dataset to load prepared images and masks. Don't forget to use augmentations.

Some of the images are 1 channels, so use `gray2rgb`.

In [100]:
def gray2rgb(img):
    if len(img.shape) != 3:
        img = np.dstack([img, img, img])
    return img

def get_iou(gt, pred):
    pred = pred > 0.5
    return (gt & pred).sum() / (gt | pred).sum()

class BirdsDataset(Dataset):
    def __init__(self, folder) -> None:
        images_folder = os.path.join(folder, 'images')
        gt_folder = os.path.join(folder, 'gt')

        self.images = []
        self.masks = []
        for class_name in os.listdir(images_folder):
            for fname in os.listdir(os.path.join(images_folder, class_name)):
                img_path = os.path.join(os.path.join(images_folder, class_name), fname)
                mask_fname = fname.rsplit('.', 1)[0] + '.png'
                mask_path = os.path.join(gt_folder, class_name, mask_fname)
                if os.path.exists(mask_path):
                    self.images.append(img_path)
                    self.masks.append(mask_path)


        self.transform = A.Compose([
            A.Resize(height=256, width=256),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.4),
            A.RandomBrightnessContrast(p=0.3),
            A.Affine(
                scale=(0.9, 1.1), translate_percent=(-0.1, 0.1), rotate=(-15, 15), p=0.5
            ),
            ToTensorV2()
        ])

    def __getitem__(self, index):
        img_path = self.images[index]
        mask_path = self.masks[index]
        img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        img = gray2rgb(img)
        img = img.astype(np.float32) / 255.0
        transformed = self.transform(image=img, mask=mask)
        transformed_img = transformed['image']
        transformed_mask = transformed['mask'].float() / 255.0

        if transformed_mask.ndim == 3 and transformed_mask.shape[0] == 1:
            transformed_mask = transformed_mask.squeeze(0)

        return transformed_img, transformed_mask

    def __len__(self):
        return len(self.images)

### Architecture (1 point)
Your task for today is to build your own Unet to solve the segmentation problem.

As an encoder, you can use pre-trained on IMAGENET models(or parts) from torchvision. The decoder must be trained from scratch.
It is forbidden to use data not from the `data` folder.

I advise you to experiment with the number of blocks so as not to overfit on the training sample and get good quality on validation.

In [102]:
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, mid_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self,x):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=True)
        x = self.block(x)
        return x

class Unet(nn.Module):
    def __init__(self):
        super().__init__()
        n_classes = 1
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.enc1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool = resnet.maxpool
        self.enc2 = resnet.layer1
        self.enc3 = resnet.layer2
        self.enc4 = resnet.layer3
        self.bottleneck = resnet.layer4
        self.dec1 = DecoderBlock(512, 256, 256)
        self.dec2 = DecoderBlock(512, 128, 128)
        self.dec3 = DecoderBlock(256, 64, 64)
        self.dec4 = DecoderBlock(128, 32, 32)

        self.final_up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(32 + 64, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        self.final_conv = nn.Conv2d(32, n_classes, kernel_size=1)


    def forward(self,x):
        e1 = self.enc1(x)
        p = self.pool(e1)
        e2 = self.enc2(p)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b  = self.bottleneck(e4)

        d1 = self.dec1(b)
        d1 = torch.cat([d1, e4], dim=1)
        d2 = self.dec2(d1)
        d2 = torch.cat([d2, e3], dim=1)
        d3 = self.dec3(d2)
        d3 = torch.cat([d3, e2], dim=1)
        d4 = self.dec4(d3)
        d4 = torch.cat([d4, e1], dim=1)
        up = self.final_up(d4)
        out = self.final_conv(up)

        return out

### Train script (0.5 point)

Complete the train and predict scripts.

In [104]:
def train_segmentation_model(data_path):
    cudnn.benchmark = True
    BATCH_SIZE = 8
    N_EPOCH = 15
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

    train_dataset = BirdsDataset(data_path + 'train')
    val_dataset = BirdsDataset(data_path + 'val')
    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                                 num_workers=4, pin_memory=True, persistent_workers=True)
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                               num_workers=4, pin_memory=True, persistent_workers=True)

    model = Unet().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCEWithLogitsLoss()
    losses_train, losses_val, ious_train, ious_val = [], [], [], []
    scaler = torch.amp.GradScaler('cuda')
    
    for epoch in range(N_EPOCH):
        model.train()

        train_loss = 0
        train_iou = 0
        train_count = 0
        
        for inputs, masks in tqdm(train_dataloader):
            inputs = inputs.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                loss = criterion(outputs, masks.unsqueeze(1))

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            pred = (torch.sigmoid(outputs) > 0.5).detach().cpu().numpy()
            v = (masks > 0.5).detach().cpu().numpy()
            batch_iou = np.mean([get_iou(i, j) for i, j in zip(v, pred)])
            train_iou += batch_iou
            train_count += 1

        losses_train.append(train_loss / train_count)
        ious_train.append(train_iou / train_count)

        model.eval()
        val_loss = 0
        val_iou = 0
        val_count = 0
        with torch.no_grad():
            for inputs, masks in tqdm(val_dataloader):
                inputs = inputs.to(DEVICE)
                masks = masks.to(DEVICE)
                outputs = model(inputs)
                loss = criterion(outputs, masks.unsqueeze(1))
                val_loss += loss.item()
                pred = (torch.sigmoid(outputs) > 0.5).detach().cpu().numpy()
                v = (masks > 0.5).detach().cpu().numpy()
                batch_iou = np.mean([get_iou(i, j) for i, j in zip(v, pred)])
                val_iou += batch_iou
                val_count += 1
        losses_val.append(val_loss / val_count)
        ious_val.append(val_iou / val_count)

        torch.save(model.state_dict(), f'model_{epoch}.pth')

        print(f"Epoch: {epoch}, train loss: {losses_train[-1]}, val loss: {losses_val[-1]}, train iou: {ious_train[-1]}, val iou: {ious_val[-1]}")

In [158]:
def predict(model, img_path):
    with torch.no_grad():
        img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        h, w = img.shape[:2]
        img = gray2rgb(img)
        img = img.astype(np.float32) / 255.0
        transform = A.Compose([
            A.Resize(height=256, width=256),
            ToTensorV2()
        ])
        transformed = transform(image=img)
        inputs = transformed['image'].unsqueeze(0).to('cuda').float()
        pred = model(inputs)
        prob = torch.sigmoid(pred).squeeze(0).squeeze(0).cpu().numpy()
        segm_small = (prob > 0.5).astype(np.uint8)
        segm = cv2.resize(segm_small, (w, h), interpolation=cv2.INTER_NEAREST)

        return segm

def get_model(path):
    model = Unet()
    model.load_state_dict(torch.load(path))
    model.eval()
    return model

In [106]:
train_segmentation_model('data/')

100%|██████████| 176/176 [00:06<00:00, 25.36it/s]


Epoch: 0, train loss: 0.17181352725010793, val loss: 0.17992449970915914, train iou: 0.6361057228754149, val iou: 0.6487286086126232


100%|██████████| 176/176 [00:06<00:00, 26.13it/s]


Epoch: 1, train loss: 0.13356470218539693, val loss: 0.13512082118541002, train iou: 0.733213509057206, val iou: 0.7311809651493067


100%|██████████| 176/176 [00:06<00:00, 26.04it/s]


...

Epoch: 12, train loss: 0.07908262586832501, val loss: 0.08024380464022132, train iou: 0.8358207929428678, val iou: 0.8355212785678685


100%|██████████| 176/176 [00:06<00:00, 25.98it/s]


Epoch: 13, train loss: 0.07801431053111912, val loss: 0.08123477188531648, train iou: 0.8390870946214374, val iou: 0.8319482288072504


100%|██████████| 176/176 [00:06<00:00, 25.93it/s]


Epoch: 14, train loss: 0.07903982162518242, val loss: 0.08090083914893595, train iou: 0.8371951789206616, val iou: 0.8353405752293216


You can also experiment with models and write a small report about results. If the report will be meaningful, you will receive an extra point.

### Testing (8 points)
Your model will be tested on the new data, similar to validation, so use techniques to prevent overfitting the model.

* IoU > 0.85 — 8 points
* IoU > 0.80 — 7 points
* IoU > 0.75 — 6 points
* IoU > 0.70 — 5 points
* IoU > 0.60 — 4 points
* IoU > 0.50 — 3 points
* IoU > 0.40 — 2 points
* IoU > 0.30 — 1 points

In [159]:
model = get_model('model_14.pth').to('cuda')

In [160]:
ious, times = [], []
test_dir = 'data/val/'

for class_name in tqdm(sorted(os.listdir(os.path.join(test_dir, 'images')))):
    for img_name in sorted(os.listdir(os.path.join(test_dir, 'images', class_name))):

        t_start = time()
        pred = predict(model, os.path.join(test_dir, 'images', class_name, img_name))
        times.append(time() - t_start)

        gt_name = img_name.replace('jpg', 'png')
        gt = np.asarray(Image.open(os.path.join(test_dir, 'gt', class_name, gt_name)), dtype = np.uint8)
        if len(gt.shape) > 2:
            gt = gt[:, :, 0]

        iou = get_iou(gt==255, pred>0.5)
        ious.append(iou)

np.mean(ious), np.mean(times)

100%|██████████| 200/200 [00:18<00:00, 10.73it/s]


(0.7526559602476695, 0.010959035811010499)

### Compression (1 point)

Try to speed up the model in any way without losing more than 1% in iou score.
For example [torch2trt](https://github.com/NVIDIA-AI-IOT/torch2trt)

In [161]:
def get_fast_model():
    model = get_model('model_14.pth')
    model = model.to('cuda')
    model.eval()
    example_input = torch.randn(1, 3, 256, 256).cuda()
    traced_model = torch.jit.trace(model, example_input)
    traced_model = torch.jit.optimize_for_inference(traced_model)
    return traced_model

In [162]:
fast_model = get_fast_model()

In [163]:
ious, times = [], []
test_dir = 'data/val/'

for class_name in tqdm(sorted(os.listdir(os.path.join(test_dir, 'images')))):
    for img_name in sorted(os.listdir(os.path.join(test_dir, 'images', class_name))):

        t_start = time()
        pred = predict(fast_model, os.path.join(test_dir, 'images', class_name, img_name))
        times.append(time() - t_start)

        gt_name = img_name.replace('jpg', 'png')
        gt = np.asarray(Image.open(os.path.join(test_dir, 'gt', class_name, gt_name)), dtype = np.uint8)
        if len(gt.shape) > 2:
            gt = gt[:, :, 0]

        iou = get_iou(gt==255, pred>0.5)
        ious.append(iou)

np.mean(ious), np.mean(times)

100%|██████████| 200/200 [00:16<00:00, 12.38it/s]


(0.7526559182401368, 0.009211334022315772)

**Bonus:** For the best iou score on test(without compression) in group you will get 1.5, 1, 0.5 extra points(for 1st, 2nd, 3rd places).